# <a id='toc1_'></a>[Title](#toc0_)
- Member 1
- Member 2
- Member 3

**Table of contents**<a id='toc0_'></a>    
  - [Import libraries and data](#toc1_1_)    
  - [Check NaNs, cast data types](#toc1_2_)    
  - [Part 1: Preprocessing the documents](#toc1_3_)    
  - [Part 2: EDA](#toc1_4_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

## <a id='toc1_1_'></a>[Import libraries and data](#toc0_)

In [1]:
import pandas as pd
from nltk.stem import PorterStemmer
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/mvelasco/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
# Read fashion products dataset
data_dir = "../../data/"
filename = "fashion_products_dataset.json"
df = pd.read_json(data_dir+filename)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28080 entries, 0 to 28079
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   _id              28080 non-null  object        
 1   actual_price     28080 non-null  object        
 2   average_rating   28080 non-null  object        
 3   brand            28080 non-null  object        
 4   category         28080 non-null  object        
 5   crawled_at       28080 non-null  datetime64[ns]
 6   description      28080 non-null  object        
 7   discount         28080 non-null  object        
 8   images           28080 non-null  object        
 9   out_of_stock     28080 non-null  bool          
 10  pid              28080 non-null  object        
 11  product_details  28080 non-null  object        
 12  seller           28080 non-null  object        
 13  selling_price    28080 non-null  object        
 14  sub_category     28080 non-null  objec

## <a id='toc1_2_'></a>[Check NaNs, cast data types](#toc0_)

In [3]:
# Check for missing values, including empty and whitespace-only strings
print("Missing values, including empty and whitespace-only strings:")
display(
    df.replace(r"^\s*$", pd.NA, regex=True)
      .isna()
      .sum()
)

# Verify product identifiers are unique
assert df["pid"].notna().all()
assert df["pid"].is_unique

Missing values, including empty and whitespace-only strings:


_id                    0
actual_price         777
average_rating      2261
brand               2009
category               0
crawled_at             0
description        11150
discount             855
images                 0
out_of_stock           0
pid                    0
product_details        0
seller              1643
selling_price          2
sub_category           0
title                  0
url                    0
dtype: int64

Before proceeding, we convert all numeric columns from strings to float. The column `out_of_stock` is already boolean, so we keep it unchanged.

In [4]:
df_old = df.copy()

# Convert prices to numeric values:
# "2,999" -> 2999; missing or invalid values -> NaN
for column in ["actual_price", "selling_price"]:
    cleaned = (
        df_old[column]
        .astype("string")
        .str.replace(",", "", regex=False)
        .str.strip()
    )

    df[column] = pd.to_numeric(cleaned, errors="raise")

# Convert discounts to percentage points:
# "69% off" -> 0.69; missing or invalid values -> NaN
df["discount"] = pd.to_numeric(
    df_old["discount"]
    .astype("string")
    .str.extract(r"^\s*(\d+(?:\.\d+)?)\s*%\s*off\s*$", expand=False)
)/100

# Convert ratings to numeric values:
# "3.9" -> 3.9; missing or invalid values -> NaN
df["average_rating"] = pd.to_numeric(
    df_old["average_rating"].astype("string").str.strip(),
    errors="coerce"
)

# Show the dataframe with the new numeric columns
numeric_cols = ["actual_price", "selling_price", "discount", "average_rating"]
print("Before:")
display(df_old[numeric_cols].head(3))
print("After:")
display(df[numeric_cols].head(3))

# Verify new data types are Float64
print("Data types of numeric fields:")
display(df[numeric_cols].dtypes)

Before:


,actual_price,selling_price,discount,average_rating
0,"2,999",921,69% off,3.9
1,"1,499",499,66% off,3.9
2,"2,999",931,68% off,3.9


After:


,actual_price,selling_price,discount,average_rating
0,2999.0,921.0,0.69,3.9
1,1499.0,499.0,0.66,3.9
2,2999.0,931.0,0.68,3.9


Data types of numeric fields:


actual_price      Float64
selling_price     Float64
discount          Float64
average_rating    Float64
dtype: object

## <a id='toc1_3_'></a>[Part 1: Preprocessing the documents](#toc0_)

In this part we create two dataframes:

1. `df` has product information *without* preprocessing. For example, in the first row ‘description’ = ‘Wool Black pants made out of 100% cotton’.
2. `index_df` only contains the fields we’ll use in the index. All fields are pre-processed (lowercase, stemmed, etc.). For example, in the first row ‘description’ = [‘wool', 'black', 'pant', 'mad', '100', 'cotton’].

The first dataframe holds the information we'll later display in the search results: the product's title, description, etc. as it was originally written. The second will be used to create the index and to perform the exploratory data analysis (EDA). We are interested in keeping the fields separate in both cases so we give each field a different weight in the index.

In [5]:
# Preprocess the text of the description column
def build_terms(text):
    """
    Preprocess text removing stop words, stemming,
    transforming in lowercase and return the tokens of the text.

    :param text: to be preprocessed
    :returns: a list of tokens corresponding to the input text after the preprocessing
    """

    stemmer = PorterStemmer()
    tokenizer = RegexpTokenizer(r'\w+')             # keep only words & numbers, drop punctuation
    stop_words = set(stopwords.words("english"))
    text= text.lower()                              # transform in lowercase
    text=  tokenizer.tokenize(text)                 # tokenize the text to get a list of terms
    text=[t for t in text if t not in stop_words]   # eliminate the stopwords
    text=[stemmer.stem(t) for t in text]            # perform stemming
    return text

test_row = df.iloc[0]
print(f"Before: {test_row['description']}")
print(f"After: {build_terms(test_row['description'])}")

Before: Yorker trackpants made from 100% rich combed cotton giving it a rich look.Designed for Comfort,Skin friendly fabric,itch-free waistband & great for all year round use Proudly made in India
After: ['yorker', 'trackpant', 'made', '100', 'rich', 'comb', 'cotton', 'give', 'rich', 'look', 'design', 'comfort', 'skin', 'friendli', 'fabric', 'itch', 'free', 'waistband', 'great', 'year', 'round', 'use', 'proudli', 'made', 'india']


In [6]:
index_df = pd.DataFrame()
index_df['pid'] = df['pid']

index_cols = ["title", "description", "brand", "category", "sub_category", "seller"]
for col in index_cols:
    index_df[col] = df[col].apply(build_terms)
index_df.head(3)

,pid,title,description,brand,category,sub_category,seller
0,TKPFCZ9EA7H5FYZH,"[solid, women, multicolor, track, pant]","[yorker, trackpant, made, 100, rich, comb, cot...",[york],"[cloth, accessori]",[bottomwear],"[shyam, enterpris]"
1,TKPFCZ9EJZV2UVRZ,"[solid, men, blue, track, pant]","[yorker, trackpant, made, 100, rich, comb, cot...",[york],"[cloth, accessori]",[bottomwear],"[shyam, enterpris]"
2,TKPFCZ9EHFCY5Z4Y,"[solid, men, multicolor, track, pant]","[yorker, trackpant, made, 100, rich, comb, cot...",[york],"[cloth, accessori]",[bottomwear],"[shyam, enterpris]"


The column `product_details` also includes some information that might be useful for the index. The problem is it is structured as a dictionary. Also, we don't want to include the dictionary keys, they'd simply add noise to our terms.

In [7]:
test_row.product_details

[{'Style Code': '1005COMBO2'},
 {'Closure': 'Elastic'},
 {'Pockets': 'Side Pockets'},
 {'Fabric': 'Cotton Blend'},
 {'Pattern': 'Solid'},
 {'Color': 'Multicolor'}]

Therefore, we get the values of the dictionary, extract the terms and add them to our `index_df`.

In [ ]:
def values_to_string(details):
    """
    Convert a product's details into a single string of its values.

    :param details: list of single-entry dicts, e.g. [{'Closure': 'Elastic'}, {'Color': 'Multicolor'}]
    :returns: a string with all the detail values joined by spaces (e.g. 'Elastic Multicolor'),
              or NaN if the input is not a list
    """
    if not isinstance(details, list):
        return pd.NA
    return " ".join(str(v) for d in details for v in d.values())

In [15]:
# Preprocess (lowercase, stem, etc.) and add to our index info 
index_df['product_details'] = df['product_details'].apply(values_to_string).apply(build_terms)
index_df.head(3)

,pid,title,description,brand,category,sub_category,seller,product_details
0,TKPFCZ9EA7H5FYZH,"[solid, women, multicolor, track, pant]","[yorker, trackpant, made, 100, rich, comb, cot...",[york],"[cloth, accessori]",[bottomwear],"[shyam, enterpris]","[1005combo2, elast, side, pocket, cotton, blen..."
1,TKPFCZ9EJZV2UVRZ,"[solid, men, blue, track, pant]","[yorker, trackpant, made, 100, rich, comb, cot...",[york],"[cloth, accessori]",[bottomwear],"[shyam, enterpris]","[1005blue, drawstr, elast, side, pocket, cotto..."
2,TKPFCZ9EHFCY5Z4Y,"[solid, men, multicolor, track, pant]","[yorker, trackpant, made, 100, rich, comb, cot...",[york],"[cloth, accessori]",[bottomwear],"[shyam, enterpris]","[1005combo4, elast, side, pocket, cotton, blen..."


In [16]:
df.to_csv(data_dir + f"fashion_products.csv", index=False)
index_df.to_csv(data_dir + f"index_info.csv", index=False)
print(f"✅ Saved 2 files to {data_dir}")

✅ Saved 2 files to ../../data/


## <a id='toc1_4_'></a>[Part 2: EDA](#toc0_)

**List of features prone to be analyzed**
* Missing rate on each field statistics --> We need to process the missing values
* Are the categories balanced? --> Explore top and low categories
* Distribution of length of title, description, final tokens
* Most common words --> This can be used for TF
* Distribution of price, ratings, discount
* Top sellers and brands regarding product quantities
* Stok